

Step 1 — Load Data

* Read CSV into Spark DataFrame
* Define schema manually (avoid automatic type inference for huge data)
* Verify columns and data types

---

Step 2 — Data Quality Check

Check:

* Missing values
* Duplicate rows
* Invalid timestamps
* Invalid user IDs/session IDs
* Invalid product IDs
* Negative or impossible amounts
* Unknown event types

---

Step 3 — Label Preparation

Convert:

outcome
none     → 0
purchase → 1

Create your target column:

label

---

Step 4 — Remove Data Leakage

Check that no feature contains future information.

Example:

* Do not use “purchase event happened after browsing” as a feature.
* Features must represent information available before prediction.

---

Step 5 — Session-Level Aggregation

Your current data is event-level:

user → session → many events

Convert it into:

one row = one session

Example:

Before:

session 101
page_view
click
product_view
add_to_cart
purchase

After:

session 101
total_events = 5
page_views = 1
clicks = 1
cart_adds = 1
products_viewed = 1
label = 1

This is the most important transformation.

---

Step 6 — Feature Extraction

Create predictive features.

From event type:

* Total number of events
* Number of page views
* Number of product views
* Number of clicks
* Number of add-to-cart events
* Number of login events

From session:

* Session duration
* First event time
* Last event time
* Number of unique products viewed
* Number of products added to cart

From product:

* Number of unique products
* Product category features (if available later)

From amount:

* Total amount viewed
* Average product amount
* Maximum product amount

---

Step 7 — Time-Based Features

Extract:

From timestamp:

* Hour of day
* Day of week
* Weekend/weekday
* Month/season

Example:

Purchase probability may differ:

2 AM vs 8 PM
weekday vs weekend

---

Step 8 — Handle Categorical Features

Your string columns:

* event type
* product ID

Need conversion.

Methods:

* One-hot encoding
* Label encoding
* Target encoding
* Embedding (advanced)

For product ID, be careful because unique IDs can create millions of features.

---

Step 9 — Handle Class Imbalance

Your data:

No purchase: 70%
Purchase: 30%

This is not extremely imbalanced, but check after session aggregation.

Options:

* Class weights
* Sampling
* Evaluation metrics like F1, AUC instead of only accuracy

---

Step 10 — Feature Vector Creation

Combine all features into one ML input:

features
[
event_count,
session_duration,
cart_add_count,
hour,
amount,
...
]

---

Step 11 — Train/Test Split

Split data:

Example:

80% training
20% testing

For timestamp data, preferably use:

past → train
future → test

not random split.

---

Step 12 — Train Models

Start with:

1. Logistic Regression (baseline)
2. Random Forest
3. Gradient Boosted Trees
4. XGBoost/LightGBM (often strong for this type of data)

---

Step 13 — Evaluate Model

Measure:

* Accuracy (not enough)
* Precision
* Recall
* F1-score
* ROC-AUC
* PR-AUC

For purchase prediction, usually focus on:

* How many actual buyers you can detect
* How many predictions are correct

---

Step 14 — Tune Model

Adjust:

* Model parameters
* Feature selection
* Threshold probability

Example:

Default:

purchase probability > 0.5

Maybe business wants:

purchase probability > 0.3

to target more users.

---

Step 15 — Deploy Prediction Pipeline

Final flow:

New User Events
       |
       v
Spark Processing
       |
       v
Feature Extraction
       |
       v
ML Model
       |
       v
Purchase Probability

---



In [ ]:
For your specific dataset (user behavior event log → purchase prediction), I would divide data cleaning + transformation into 12 steps.

Not generic cleaning. These are the actual functions/operations your pipeline should perform.

⸻

Step 1: Load and inspect raw events

Function:

load_data()

Purpose:

* Read CSV
* Check first events
* Understand event flow

Output:

raw_event_dataframe

⸻

Step 2: Standardize column names

Function:

standardize_columns()

Example:

Before:

User ID
Session ID
Event Type

After:

user_id
session_id
event_type

⸻

Step 3: Fix data types

Function:

cast_data_types()

Convert:

user_id       → integer/string
session_id    → string
timestamp     → timestamp
amount        → double
event_type    → string

⸻

Step 4: Clean event types

Function:

clean_event_type()

Operations:

"Page View"
"page_view"
"PAGE VIEW"

Convert:

page_view

Also remove unknown events:

Example:

event_type = "abc123"

Remove or mark as unknown.

⸻

Step 5: Validate user and session identity

Function:

validate_user_session()

Rules:

Remove:

user_id = null
session_id = null

Check:

same user
same session
correct relationship

⸻

Step 6: Clean timestamp sequence

Function:

clean_timestamp()

Rules:

Remove:

invalid timestamp
future timestamp

Sort:

user_id
    |
session_id
    |
timestamp

Example:

Before:

10:05 purchase
10:01 view

After:

10:01 view
10:05 purchase

⸻

Step 7: Clean amount column

Function:

clean_amount()

Rules:

Purchase:

amount must exist
amount > 0

Non-purchase:

amount can be null

Remove:

amount = -500
amount = "abc"

⸻

Step 8: Clean outcome label

Function:

create_target_label()

Convert:

Before:

purchase
null

After:

1
0

Example:

purchased = 1
not purchased = 0

⸻

Step 9: Remove impossible user journeys

Function:

validate_event_sequence()

Examples:

Remove or fix:

purchase
   ↓
page_view

because purchase cannot happen before viewing.

Detect:

logout before login

⸻

Step 10: Handle duplicate events

Function:

remove_duplicate_events()

Example:

Same:

user_id
session_id
timestamp
event_type
product_id

appearing multiple times.

Decide:

Remove or keep depending on meaning.

⸻

Step 11: Aggregate event logs into user features

Function:

create_behavior_features()

Transform:

From:

1000 event rows

To:

1 user = 1 row

Create:

total_views
total_clicks
total_cart_adds
total_sessions
unique_products
avg_session_time

⸻

Step 12: Prepare ML dataset

Function:

prepare_training_dataset()

Final structure:

user_id
features:
    view_count
    click_count
    cart_count
    session_count
    product_count
    avg_amount
label:
    purchased

Ready for:

ML model training

⸻

Your complete pipeline becomes:

Raw CSV
   ↓
load_data()
   ↓
standardize_columns()
   ↓
cast_data_types()
   ↓
clean_event_type()
   ↓
validate_user_session()
   ↓
clean_timestamp()
   ↓
clean_amount()
   ↓
create_target_label()
   ↓
validate_event_sequence()
   ↓
remove_duplicate_events()
   ↓
create_behavior_features()
   ↓
prepare_training_dataset()
   ↓
Machine Learning Model

For your use case, these 12 steps are enough. You don’t need hundreds of generic data-cleaning rules. The important part is creating a correct behavioral representation before training.